In [4]:
import os
# Ensure the script uses GPU 5 explicitly, and CUDA detects ONLY GPU #5
os.environ["CUDA_VISIBLE_DEVICES"] = "5"

import json
import torch
import networkx as nx
from omegaconf import OmegaConf
from model import GraphTrajectoryLM
from tokenizer_utils import load_tokenizer

CHECKPOINT_PATH = "checkpoints/last.ckpt"
TOKENIZER_DIR = "data_output/tokenizer"
GRAPH_PATH = "data_output/graph.json"
CONFIG_PATH = "conf/config.yaml"

# In this environment, cuda:0 is now mapped to GPU 5 on the machine
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"

# Load tokenizer
tokenizer = load_tokenizer(TOKENIZER_DIR)
print(f"Vocab size: {tokenizer.vocab_size}")
print(f"Pad ID: {tokenizer.pad_token_id}, EOS ID: {tokenizer.eos_token_id}")

# Load config and construct model with matching architecture, then load weights
cfg = OmegaConf.load(CONFIG_PATH)
model = GraphTrajectoryLM(
    vocab_size=tokenizer.vocab_size,
    pad_token_id=tokenizer.pad_token_id,
    eos_token_id=tokenizer.eos_token_id,
    model_config=cfg.model,
    train_config=cfg.train,
)

ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt["state_dict"])
model.eval()
model.to(DEVICE)
print(f"Model loaded on {DEVICE}")

# Load graph
with open(GRAPH_PATH) as f:
    graph_data = json.load(f)

G = nx.Graph()
for node, neighbors in graph_data["adjacency"].items():
    for nb in neighbors:
        G.add_edge(int(node), int(nb))

print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

Vocab size: 105
Pad ID: 0, EOS ID: 1
Model loaded on cuda:0
Graph: 100 nodes, 474 edges


In [5]:
@torch.no_grad()
def generate_trajectory(
    start_node: int,
    goal_node: int,
    max_new_tokens: int = 50,
    temperature: float = 1.0,
    top_k: int = 0,
    top_p: float = 1.0,
):
    """Generate a trajectory from start_node to goal_node using the trained model.

    Returns the decoded trajectory string and the list of generated node tokens.
    """
    prompt = f"<start_goal> {start_node} {goal_node} <end_goal>"
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(DEVICE)

    eos_id = tokenizer.eos_token_id
    pad_id = tokenizer.pad_token_id

    generated = input_ids
    for _ in range(max_new_tokens):
        outputs = model.model(input_ids=generated)
        logits = outputs.logits[:, -1, :] / temperature

        # Never sample pad
        logits[:, pad_id] = -float("inf")

        if top_k > 0:
            topk_vals, _ = torch.topk(logits, top_k)
            logits[logits < topk_vals[:, -1:]] = -float("inf")

        if top_p < 1.0:
            sorted_logits, sorted_idx = torch.sort(logits, descending=True)
            cumulative_probs = torch.cumsum(torch.softmax(sorted_logits, dim=-1), dim=-1)
            remove = cumulative_probs - torch.softmax(sorted_logits, dim=-1) >= top_p
            sorted_logits[remove] = -float("inf")
            logits = sorted_logits.scatter(1, sorted_idx, sorted_logits)

        probs = torch.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        generated = torch.cat([generated, next_token], dim=-1)

        if next_token.item() == eos_id:
            break

    decoded = tokenizer.decode(generated[0], skip_special_tokens=False)
    path_tokens = tokenizer.decode(generated[0, input_ids.shape[1]:], skip_special_tokens=True).split()
    return decoded, path_tokens

In [7]:
def validate_trajectory(start: int, goal: int, path_tokens: list[str]) -> dict:
    """Check whether a generated trajectory is a valid path in the graph."""
    try:
        path_nodes = [int(t) for t in path_tokens if t not in ("<eos>",)]
    except ValueError:
        return {"valid": False, "reason": "non-integer tokens in path", "path": path_tokens}

    if not path_nodes:
        return {"valid": False, "reason": "empty path", "path": []}

    if path_nodes[0] != start:
        return {"valid": False, "reason": f"path starts at {path_nodes[0]}, expected {start}", "path": path_nodes}

    reached_goal = path_nodes[-1] == goal

    for i in range(len(path_nodes) - 1):
        if not G.has_edge(path_nodes[i], path_nodes[i + 1]):
            return {
                "valid": False,
                "reason": f"no edge {path_nodes[i]}->{path_nodes[i+1]}",
                "path": path_nodes,
                "reached_goal": False,
            }

    return {"valid": True, "reached_goal": reached_goal, "path": path_nodes, "length": len(path_nodes)}

In [8]:
# --- Generate some trajectories ---
import random

nodes = list(G.nodes())
random.seed(42)

print("=" * 80)
print("Sampling trajectories (temperature=0.8, top_k=20)")
print("=" * 80)

for i in range(10):
    start, goal = random.sample(nodes, 2)
    shortest = nx.shortest_path_length(G, start, goal)
    decoded, path_tokens = generate_trajectory(start, goal, temperature=0.8, top_k=20)
    result = validate_trajectory(start, goal, path_tokens)

    status = "VALID" if result["valid"] else "INVALID"
    reached = " (reached goal)" if result.get("reached_goal") else " (did NOT reach goal)"
    print(f"\n[{i+1}] {start} -> {goal}  (shortest={shortest})")
    print(f"  Generated: {' '.join(path_tokens)}")
    print(f"  {status}{reached}")
    if not result["valid"]:
        print(f"  Reason: {result['reason']}")

Sampling trajectories (temperature=0.8, top_k=20)

[1] 88 -> 1  (shortest=2)
  Generated: 88 29 1
  VALID (reached goal)

[2] 10 -> 23  (shortest=3)
  Generated: 10 37 50 23
  VALID (reached goal)

[3] 9 -> 52  (shortest=3)
  Generated: 9 4 96 52
  VALID (reached goal)

[4] 93 -> 34  (shortest=1)
  Generated: 93 31 76 80 71 67 34
  VALID (reached goal)

[5] 23 -> 95  (shortest=3)
  Generated: 23 35 16 59 16 59 28 15 28 59
  VALID (did NOT reach goal)

[6] 47 -> 23  (shortest=4)
  Generated: 47 18 85 94 85 94 85 94 85 94 86 69 31 66 74 49 45 82 23 1 1
  INVALID (did NOT reach goal)
  Reason: no edge 66->74

[7] 72 -> 89  (shortest=2)
  Generated: 72 25 89
  VALID (reached goal)

[8] 94 -> 56  (shortest=2)
  Generated: 94 85 94 31 76 73 91 77 75 52 51 7 3 34 37 56
  VALID (reached goal)

[9] 13 -> 10  (shortest=2)
  Generated: 13 0 10
  VALID (reached goal)

[10] 89 -> 75  (shortest=1)
  Generated: 89 75
  VALID (reached goal)


In [7]:
# --- Batch evaluation: accuracy stats ---
num_trials = 100
valid_count = 0
reached_goal_count = 0
valid_and_reached = 0

random.seed(123)
for _ in range(num_trials):
    s, g = random.sample(nodes, 2)
    _, path_tokens = generate_trajectory(s, g, temperature=0.8, top_k=20)
    result = validate_trajectory(s, g, path_tokens)
    if result["valid"]:
        valid_count += 1
        if result.get("reached_goal"):
            valid_and_reached += 1
    if result.get("reached_goal"):
        reached_goal_count += 1

print(f"Results over {num_trials} random (start, goal) pairs:")
print(f"  Valid paths:            {valid_count}/{num_trials} ({100*valid_count/num_trials:.1f}%)")
print(f"  Reached goal:           {reached_goal_count}/{num_trials} ({100*reached_goal_count/num_trials:.1f}%)")
print(f"  Valid + reached goal:   {valid_and_reached}/{num_trials} ({100*valid_and_reached/num_trials:.1f}%)")

Results over 100 random (start, goal) pairs:
  Valid paths:            83/100 (83.0%)
  Reached goal:           62/100 (62.0%)
  Valid + reached goal:   62/100 (62.0%)


In [46]:
# --- Interactive: generate your own trajectory ---
# Change start_node and goal_node to whatever you like, then run this cell.

start_node = 5
goal_node = 2

decoded, path_tokens = generate_trajectory(start_node, goal_node, temperature=0.8, top_k=20)
result = validate_trajectory(start_node, goal_node, path_tokens)

print(f"Prompt:    <start_goal> {start_node} {goal_node} <end_goal>")
print(f"Generated: {decoded}")
print(f"Path:      {' -> '.join(path_tokens)}")
print(f"Valid:     {result['valid']}")
print(f"Reached:   {result.get('reached_goal', False)}")
if not result["valid"]:
    print(f"Reason:    {result['reason']}")
else:
    shortest = nx.shortest_path_length(G, start_node, goal_node)
    print(f"Path len:  {result['length']}  (shortest = {shortest})")

Prompt:    <start_goal> 5 2 <end_goal>
Generated: <start_goal> 5 2 <end_goal> 5 22 66 13 51 32 90 36 70 73 84 52 51 52 56 7 19 57 56 44 30 40 12 39 57 <eos>
Path:      5 -> 22 -> 66 -> 13 -> 51 -> 32 -> 90 -> 36 -> 70 -> 73 -> 84 -> 52 -> 51 -> 52 -> 56 -> 7 -> 19 -> 57 -> 56 -> 44 -> 30 -> 40 -> 12 -> 39 -> 57
Valid:     False
Reached:   False
Reason:    no edge 84->52
